# 3D Gaussian Splatting on Mip-NeRF 360 — Colab Demo

This notebook reproduces a minimal, single-scene run of the original 3D Gaussian Splatting pipeline (Kerbl et al., SIGGRAPH 2023) on one scene of the **Mip-NeRF 360** dataset, using the official `graphdeco-inria/gaussian-splatting` reference implementation.

**Goal.** Train 3DGS on the `bonsai` scene and report:
- PSNR / SSIM / LPIPS on the test split
- Wall-clock training time
- Final number of Gaussians
- Inference FPS at test resolution

These numbers are then compared with Table 1 of the original 3DGS paper as a sanity check.

**Why `bonsai`.** It is one of the smallest Mip-NeRF 360 scenes (~290 images, indoor, contained geometry), so it trains quickly on a free Colab T4 and avoids the VRAM spikes that the unbounded outdoor scenes (`garden`, `bicycle`, ...) produce at full resolution.

**Runtime requirements.** Set the Colab runtime to `Runtime → Change runtime type → GPU` (T4 is enough, A100 is faster). The training step takes roughly 30–45 minutes on a T4.

## 1. Environment check

In [ ]:
!nvidia-smi

Wed Apr  8 13:50:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch, sys
print('Python :', sys.version.split()[0])
print('Torch  :', torch.__version__)
print('CUDA   :', torch.version.cuda)
print('Device :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Python : 3.12.13
Torch  : 2.10.0+cu128
CUDA   : 12.8
Device : Tesla T4


## 2. Clone the official 3DGS repository

We use the original Inria implementation. The `--recursive` flag is mandatory because the rasterizer and the simple-knn module live in submodules.

In [ ]:
%cd /content
![ -d gaussian-splatting ] || git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
%cd gaussian-splatting
!git log -1 --oneline

/content
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 1053, done.
remote: Total 1053 (delta 0), reused 0 (delta 0), pack-reused 1053 (from 1)
Receiving objects: 100% (1053/1053), 78.71 MiB | 20.02 MiB/s, done.
Resolving deltas: 100% (595/595), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects:

## 3. Install dependencies

We install the Python dependencies and then build the two CUDA extensions (`diff-gaussian-rasterization` and `simple-knn`).

If the build fails because of a CUDA / PyTorch ABI mismatch, the most common fix on Colab is to pin a compatible PyTorch (e.g. `torch==2.3.1+cu121`) before running this cell.

In [ ]:
!pip -q install plyfile tqdm opencv-python lpips ninja

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 11.6 MB/s eta 0:00:00


In [ ]:
%cd /content/gaussian-splatting
!pip -q install submodules/diff-gaussian-rasterization
!pip -q install submodules/simple-knn
# fused-ssim is optional but speeds up the SSIM loss; skip if it fails to build.
!pip -q install submodules/fused-ssim || echo 'fused-ssim not built (optional)'

/content/gaussian-splatting
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


## 4. Download the `bonsai` scene

The Mip-NeRF 360 dataset is hosted by the original authors as **one single archive** for the four indoor scenes (`360_v2.zip`, ~8 GB). Individual scenes are not downloadable, so we grab the full archive with `wget` and then extract only the `bonsai/` subfolder, deleting the zip afterwards to free up Colab disk.

Each extracted scene contains the COLMAP reconstruction (`sparse/`) and the images at several resolutions (`images/`, `images_2/`, `images_4/`, `images_8/`).

In [ ]:
import os, zipfile, pathlib, subprocess

DATA_ROOT  = pathlib.Path('/content/data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVE    = DATA_ROOT / '360_v2.zip'
SCENE_DIR  = DATA_ROOT / 'bonsai'
ARCHIVE_URL = 'https://storage.googleapis.com/gresearch/refraw360/360_v2.zip'

if not SCENE_DIR.exists():
    if not ARCHIVE.exists():
        print('Downloading 360_v2.zip (~8 GB) ...')
        # wget is much more reliable than urllib for large Google CDN downloads
        subprocess.run(['wget', '-q', '--show-progress', '-O', str(ARCHIVE), ARCHIVE_URL], check=True)
    print('Extracting only the bonsai/ entries ...')
    with zipfile.ZipFile(ARCHIVE, 'r') as z:
        members = [m for m in z.namelist() if m.startswith('bonsai/')]
        print(f'  {len(members)} files to extract')
        z.extractall(DATA_ROOT, members=members)
    print('Removing the archive to free disk space ...')
    ARCHIVE.unlink()

print('Scene contents:')
for p in sorted(SCENE_DIR.iterdir()):
    print(' -', p.name)

Extracting only the bonsai/ entries ...
  1179 files to extract
Removing the archive to free disk space ...
Scene contents:
 - images
 - images_2
 - images_4
 - images_8
 - poses_bounds.npy
 - sparse


## 5. Train 3D Gaussian Splatting

We launch the official `train.py` with the `--eval` flag, which enables the Mip-NeRF 360 train/test split (every 8th view goes into the test set), and use the `images_2/` (half-resolution) images to keep memory and time manageable on a T4.

On a T4 the run takes roughly 30–45 minutes for the full 30k iterations. Set `ITERS = 7000` for a quick smoke-test (~10 min) or keep `30000` for the reference numbers.

In [ ]:
import time, subprocess, json, pathlib

ITERS    = 30000
OUT_DIR  = pathlib.Path('/content/gaussian-splatting/output/bonsai')
OUT_DIR.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    'python', 'train.py',
    '-s', '/content/data/bonsai',
    '-m', str(OUT_DIR),
    '--eval',
    '--resolution', '2',           # use the images_2/ folder (half-res)
    '--iterations', str(ITERS),
    '--test_iterations', '7000', '15000', '30000',
    '--save_iterations',  '7000', '15000', '30000',
]

print('Running:', ' '.join(cmd))
t0 = time.time()
ret = subprocess.run(cmd, cwd='/content/gaussian-splatting')
train_time_s = time.time() - t0
print(f'\nTraining wall time: {train_time_s/60:.1f} min  (return code {ret.returncode})')

# Persist the wall-clock so the summary cell can pick it up.
with open('/content/train_time.json', 'w') as f:
    json.dump({'train_time_s': train_time_s, 'iterations': ITERS}, f)

Running: python train.py -s /content/data/bonsai -m /content/gaussian-splatting/output/bonsai --eval --resolution 2 --iterations 30000 --test_iterations 7000 15000 30000 --save_iterations 7000 15000 30000

Training wall time: 67.1 min  (return code 0)


## 6. Render the test split

In [ ]:
!python render.py -m {OUT_DIR}

Looking for config file in /content/gaussian-splatting/output/bonsai/cfg_args
Config file found: /content/gaussian-splatting/output/bonsai/cfg_args
Rendering /content/gaussian-splatting/output/bonsai
Loading trained model at iteration 30000 [08/04 15:12:34]
------------LLFF HOLD------------- [08/04 15:12:36]
Reading camera 292/292 [08/04 15:12:36]
Loading Training Cameras [08/04 15:12:36]
Loading Test Cameras [08/04 15:13:25]
Rendering progress: 100% 255/255 [06:08<00:00,  1.45s/it]
Rendering progress: 100% 37/37 [00:53<00:00,  1.44s/it]


## 7. Compute PSNR / SSIM / LPIPS on the test views

The official `metrics.py` script loops over the test views rendered in step 6 and writes the metrics to `<output>/results.json`. We only need PSNR for the report, but SSIM and LPIPS come for free.

In [ ]:
!python metrics.py -m {OUT_DIR}

import json, pathlib
results_path = pathlib.Path(OUT_DIR) / 'results.json'
if results_path.exists():
    metrics = json.loads(results_path.read_text())
    print(json.dumps(metrics, indent=2))
else:
    print('results.json not found — check that render and metrics steps succeeded')


Scene: /content/gaussian-splatting/output/bonsai
Method: ours_30000
Metric evaluation progress:   0% 0/37 [00:00<?, ?it/s]Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth

  0% 0.00/528M [00:00<?, ?B/s]
  2% 10.8M/528M [00:00<00:04, 113MB/s]
  5% 27.2M/528M [00:00<00:03, 148MB/s]
  9% 45.0M/528M [00:00<00:03, 165MB/s]
 12% 61.5M/528M [00:00<00:02, 167MB/s]
 15% 79.9M/528M [00:00<00:02, 176MB/s]
 18% 96.8M/528M [00:00<00:02, 172MB/s]
 22% 116M/528M [00:00<00:02, 183MB/s] 
 25% 134M/528M [00:00<00:02, 183MB/s]
 29% 151M/528M [00:00<00:02, 178MB/s]
 32% 169M/528M [00:01<00:02, 181MB/s]
 35% 187M/528M [00:01<00:01, 183MB/s]
 39% 205M/528M [00:01<00:01, 184MB/s]
 42% 223M/528M [00:01<00:01, 180MB/s]
 45% 240M/528M [00:01<00:01, 175MB/s]
 49% 257M/528M [00:01<00:01, 177MB/s]
 52% 275M/528M [00:01<00:01, 179MB/s]
 56% 294M/528M [00:01<00:01, 186MB/s]
 60% 316M/528M [00:01<00:01, 197MB/s]
 63% 334M/528M [00:01<00:01

## 8. Number of Gaussians and inference FPS

The trained model is a single `.ply` file under `<output>/point_cloud/iteration_<ITERS>/point_cloud.ply`. The number of points in that file is the final Gaussian count.

For FPS we re-render every test view in a tight loop and time the render call. This matches the procedure used in the 3DGS paper.

In [ ]:
import sys, time, pathlib, torch
sys.path.insert(0, '/content/gaussian-splatting')

from plyfile import PlyData

ply_path = pathlib.Path(OUT_DIR) / f'point_cloud/iteration_{ITERS}/point_cloud.ply'
n_gauss  = len(PlyData.read(str(ply_path))['vertex'])
print(f'Number of Gaussians: {n_gauss:,}')

Number of Gaussians: 1,067,487


In [ ]:
# Inference FPS: load the trained model and re-render every test view in a loop.
import sys, time, torch
sys.path.insert(0, '/content/gaussian-splatting')

from argparse import Namespace
from scene import Scene
from gaussian_renderer import render
from scene.gaussian_model import GaussianModel

# Re-build the same model/pipeline argument objects used by render.py.
# We replicate every field that ModelParams / PipelineParams expose, even those
# we don't actively use, because Scene / render touch them by name.
model_args = Namespace(
    sh_degree=3,
    source_path='/content/data/bonsai',
    model_path=str(OUT_DIR),
    images='images',
    depths='',
    resolution=2,
    white_background=False,
    data_device='cuda',
    eval=True,
    train_test_exp=False,
)
pipe_args = Namespace(
    convert_SHs_python=False,
    compute_cov3D_python=False,
    debug=False,
    antialiasing=False,
)

gaussians = GaussianModel(model_args.sh_degree)
scene     = Scene(model_args, gaussians, load_iteration=ITERS, shuffle=False)
background = torch.tensor([0., 0., 0.], dtype=torch.float32, device='cuda')

test_cams = scene.getTestCameras()
print(f'Test views: {len(test_cams)}')

# Warm-up.
for cam in test_cams[:3]:
    _ = render(cam, gaussians, pipe_args, background)
torch.cuda.synchronize()

# Timed loop.
N_REPEAT = 5
torch.cuda.synchronize()
t0 = time.time()
for _ in range(N_REPEAT):
    for cam in test_cams:
        _ = render(cam, gaussians, pipe_args, background)
torch.cuda.synchronize()
elapsed = time.time() - t0

n_frames = N_REPEAT * len(test_cams)
fps = n_frames / elapsed
print(f'Rendered {n_frames} frames in {elapsed:.2f} s  \u2192  {fps:.1f} FPS')

Loading trained model at iteration 30000
------------LLFF HOLD-------------
Reading camera 292/292
Loading Training Cameras
Loading Test Cameras
Test views: 37
Rendered 185 frames in 4.31 s  →  42.9 FPS


## 9. Final summary table

Collects everything in one place so the numbers can be copy-pasted into the report.

In [ ]:
import json, pathlib

results = json.loads((pathlib.Path(OUT_DIR) / 'results.json').read_text())
# `results` is a dict keyed by iteration directory; pick the last one.
key = sorted(results.keys())[-1]
psnr  = results[key]['PSNR']
ssim  = results[key]['SSIM']
lpips = results[key]['LPIPS']

tt = json.loads(open('/content/train_time.json').read())['train_time_s']

print('========== 3DGS on Mip-NeRF 360 / bonsai ==========')
print(f'Iterations         : {ITERS}')
print(f'Train wall-time    : {tt/60:6.1f} min')
print(f'#Gaussians (final) : {n_gauss:>10,}')
print(f'PSNR               : {psnr:6.2f}')
print(f'SSIM               : {ssim:6.3f}')
print(f'LPIPS              : {lpips:6.3f}')
print(f'Inference FPS      : {fps:6.1f}')

========== 3DGS on Mip-NeRF 360 / bonsai ==========
Iterations         : 30000
Train wall-time    :   67.1 min
#Gaussians (final) :  1,067,487
PSNR               :  32.44
SSIM               :  0.948
LPIPS              :  0.173
Inference FPS      :   42.9


## 10. Reference numbers

From Table 1 of Kerbl et al., *3D Gaussian Splatting for Real-Time Radiance Field Rendering* (SIGGRAPH 2023), evaluated on the Mip-NeRF 360 indoor scenes at 30k iterations on an NVIDIA A6000:

| Scene  | PSNR | SSIM | LPIPS | Train (min) | FPS  |
|--------|------|------|-------|-------------|------|
| bonsai | 32.20 | 0.941 | 0.205 | ~30 | ~150 |

Our Colab T4 numbers should land close to PSNR/SSIM/LPIPS (these are scene-intrinsic, not hardware-dependent). Training time and FPS will be slower than the A6000 reference because the T4 is a much weaker GPU.